# **Legislative Measures Classification Notebook**

This notebook processes the dataset of U.S. congressional bills for the 119th Congress.
It performs the following steps:

1. Loads the raw dataset and classifies each record according to its legislative prefix.  
2. Cleans the `classification` column to make it human-readable (`Bill`, `Resolution`).  
3. Saves and provides the cleaned dataset for downstream use.

The datasets processed here correspond to the **119th U.S. Congress** and were downloaded from the public data portal at [Plural Policy](https://americaspoliticalpulse.com/data/).

- **Download instructions:**

1. Scroll down to United States on the List (*United States 119th Congress*). Or download directly [Here](https://data.openstates.org/csv/latest/US_119_csv_2bmRrtyLymbbJfXI2C8Hqz.zip).

2. Go to US --> 119 --> US_119_bill.csv


## **Load and classify bill identifiers**

This section loads the source CSV file, identifies each record’s legislative prefix
(e.g., HR, S, HRES, SJRES), and creates two new columns:
- `prefix_category` — the prefix extracted from the identifier.
- `measure_type` — the corresponding legislative measure type.

In [ ]:
import pandas as pd
import re

# Define relative paths
input_path = (
    "/content/drive/Shareddrives/Data Archive/Bridge_Grades/Input_Files/Snapshot_Raw_2025_11_17/Raw_A_2025_11_17/US_119_bills.csv"
)
path_status_bills = "/content/drive/Shareddrives/Data Archive/Bridge_Grades/Input_Files/Snapshot_Raw_2025_11_17/Raw_A_2025_11_17/US_119_bill_actions.csv"

df_status_bills = pd.read_csv(path_status_bills)
# Load the dataset
df = pd.read_csv(input_path)

# Define patterns for classification
patterns = {
    r"^HR\s*\d+": ("HR", "Bill"),
    r"^S\s*\d+": ("S", "Bill"),
    r"^HJRES\s*\d+": ("HJRES", "Joint Resolution"),
    r"^SJRES\s*\d+": ("SJRES", "Joint Resolution"),
    r"^HCONRES\s*\d+": ("HCONRES", "Concurrent Resolution"),
    r"^SCONRES\s*\d+": ("SCONRES", "Concurrent Resolution"),
    r"^HRES\s*\d+": ("HRES", "Simple Resolution"),
    r"^SRES\s*\d+": ("SRES", "Simple Resolution"),
}

# Function to classify identifiers
def classify_identifier(identifier):
    if pd.isna(identifier):
        return ("Unknown", "Unknown")
    identifier = str(identifier).strip().upper()
    for pattern, labels in patterns.items():
        if re.match(pattern, identifier):
            return labels
    return ("Unknown", "Unknown")

# Apply classification
df[["prefix_category", "measure_type"]] = df["identifier"].apply(
    lambda x: pd.Series(classify_identifier(x))
)

print("Sample classified records:")
df[["identifier", "prefix_category", "measure_type"]].head()


Sample classified records:


,identifier,prefix_category,measure_type
0,HR 1776,HR,Bill
1,S 693,S,Bill
2,S 690,S,Bill
3,HR 693,HR,Bill
4,HR 1806,HR,Bill


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **Clean the `classification` column and export results**

This section converts list-style text values such as `['bill']` or `['resolution']`
into simple readable labels (`Bill`, `Resolution`).  
It then saves the resulting dataset to the preprocessing output directory.


In [ ]:
import ast

# Function to clean the classification column
def clean_classification(value):
    if pd.isna(value):
        return "Unknown"
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list) and parsed:
            val = parsed[0].strip().lower()
            if val == "bill":
                return "Bill"
            elif val == "resolution":
                return "Resolution"
        return "Unknown"
    except (ValueError, SyntaxError):
        return "Unknown"

# Apply cleaning
df["classification"] = df["classification"].apply(clean_classification)


In [ ]:
df.head()

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,prefix_category,measure_type
0,ocd-bill/223e43bd-1056-4b71-ab60-4079f89af9d5,HR 1776,New Health Options Act of 2025,Bill,[],119,United States,lower,HR,Bill
1,ocd-bill/757c0231-d2d1-408a-a06a-202e3d0abef1,S 693,DHS Better Ballistic Body Armor Act,Bill,[],119,United States,upper,S,Bill
2,ocd-bill/be153c8b-cb6d-4bc2-a4e8-b4f2474a309a,S 690,Overdose RADAR Act,Bill,[],119,United States,upper,S,Bill
3,ocd-bill/aae01c67-1682-43b0-b4d5-86862e3dc2d7,HR 693,Restoring Confidence in the World Anti-Doping ...,Bill,[],119,United States,lower,HR,Bill
4,ocd-bill/c3944c75-caca-4e65-ae2d-910575f0ba68,HR 1806,Triple-Negative Breast Cancer Research and Edu...,Bill,[],119,United States,lower,HR,Bill


In [ ]:
df_status_bills.head()

,id,bill_id,organization_id,description,date,classification,order
0,1e5d6e4e-133c-4d86-8da6-ce3dc6122812,ocd-bill/1bcba77f-ab73-406d-b22c-13605d6698a8,ocd-organization/24af4233-d9b5-5933-91b2-51d29...,Ordered to be Reported (Amended) by Voice Vote.,2025-02-26T05:00:00+00:00,[],0
1,67611bfe-087f-430c-9ce0-ace5ba4ee2b1,ocd-bill/0870144d-70b6-47d3-8f55-f6ad0ebda8e6,ocd-organization/24af4233-d9b5-5933-91b2-51d29...,Ordered to be Reported by Voice Vote.,2025-02-26T05:00:00+00:00,[],0
2,5d1831ba-cc7c-4bc3-b324-b30bb56f5761,ocd-bill/d94689f7-7970-43b8-84d5-329bdaef1dc2,ocd-organization/24af4233-d9b5-5933-91b2-51d29...,Ordered to be Reported by Voice Vote.,2025-02-26T05:00:00+00:00,[],0
3,96b6a9ec-7e38-4ff0-8a3b-86fd1a603ba9,ocd-bill/06bb5568-bb6a-4dc7-9654-a6f11bee0534,ocd-organization/24af4233-d9b5-5933-91b2-51d29...,Ordered to be Reported (Amended) by the Yeas a...,2025-02-26T05:00:00+00:00,[],0
4,b5d23d0d-dfb6-4732-975c-c12a0e10ae3d,ocd-bill/5b5b2946-4fea-417a-9245-15aeebe1c5d4,ocd-organization/24af4233-d9b5-5933-91b2-51d29...,Referred to the House Committee on Foreign Aff...,2025-03-06T05:00:00+00:00,['referral-committee'],0


In [ ]:
import ast

df_status_bills['classification_clean'] = df_status_bills['classification'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)


In [ ]:
df_became_law = df_status_bills[
    df_status_bills['classification_clean'].apply(lambda lst: 'became-law' in lst)
]


In [ ]:
df_became_law

,id,bill_id,organization_id,description,date,classification,order,classification_clean
1416,07e36971-0723-43f8-a211-71929affe64b,ocd-bill/47448845-eb89-4fb5-aea9-a2f70a1fd079,ocd-organization/16b52117-31a3-5f05-96a6-fd580...,Became Public Law No: 119-2.,2025-03-14T04:00:00+00:00,['became-law'],0,[became-law]
1417,9bcf7aa3-9c17-4aba-938e-fcc553dd606c,ocd-bill/e36cb349-1dc3-4a9a-a4f8-e7c89561299b,ocd-organization/16b52117-31a3-5f05-96a6-fd580...,Became Public Law No: 119-3.,2025-03-14T04:00:00+00:00,['became-law'],0,[became-law]
1418,8381a880-b461-4f11-a69b-26add0fcc2b6,ocd-bill/ec3ed283-d0f2-45ff-852f-f1462b8b5162,ocd-organization/16b52117-31a3-5f05-96a6-fd580...,Became Public Law No: 119-4.,2025-03-15T04:00:00+00:00,['became-law'],0,[became-law]
2207,5d9b724f-fe3e-47c8-b4f9-b8cded9b03c1,ocd-bill/e8e8cdca-83bf-434d-9a3b-3df5b7291da3,ocd-organization/16b52117-31a3-5f05-96a6-fd580...,Became Public Law No: 119-11.,2025-05-09T04:00:00+00:00,['became-law'],0,[became-law]
2209,11af4cb5-4c70-4b43-9237-2e93d1a6fb83,ocd-bill/5c0fb17e-a3dd-46ea-b979-1be654b6c457,ocd-organization/16b52117-31a3-5f05-96a6-fd580...,Became Public Law No: 119-8.,2025-05-09T04:00:00+00:00,['became-law'],0,[became-law]
2216,d23d35e9-f3af-403d-aa97-6a36b69ed78c,ocd-bill/30325e59-e9c1-4faa-951c-d56b5a89eece,ocd-organization/16b52117-31a3-5f05-96a6-fd580...,Became Public Law No: 119-6.,2025-05-09T04:00:00+00:00,['became-law'],0,[became-law]
2217,22f8349b-00b6-45e9-9c27-460a3a4701b0,ocd-bill/d14e801e-90b4-4459-96ba-abcc08cd8b54,ocd-organization/16b52117-31a3-5f05-96a6-fd580...,Became Public Law No: 119-10.,2025-05-09T04:00:00+00:00,['became-law'],0,[became-law]
2218,2d36a867-2fae-4a3c-83af-5f729a36b153,ocd-bill/f56c4d76-0185-4493-932a-5894e6dbddcb,ocd-organization/16b52117-31a3-5f05-96a6-fd580...,Became Public Law No: 119-7.,2025-05-09T04:00:00+00:00,['became-law'],0,[became-law]
2219,4d1d524e-dec0-4814-8bec-e01f4b952d3f,ocd-bill/2c5607ad-897c-4c3e-ae25-0c8fa388c631,ocd-organization/16b52117-31a3-5f05-96a6-fd580...,Became Public Law No: 119-9.,2025-05-09T04:00:00+00:00,['became-law'],0,[became-law]
2674,54a56ddb-5d0f-4bfd-b852-66bba7a18b23,ocd-bill/c4af9545-fcaa-4ecb-aa68-b7a65deadf46,ocd-organization/16b52117-31a3-5f05-96a6-fd580...,Became Public Law No: 119-13.,2025-05-23T04:00:00+00:00,['became-law'],0,[became-law]


In [ ]:
df = df.merge(df_became_law[['bill_id', 'classification_clean']], left_on='id',right_on='bill_id', how='left')

In [ ]:
df.head()

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,prefix_category,measure_type,bill_id,classification_clean
0,ocd-bill/223e43bd-1056-4b71-ab60-4079f89af9d5,HR 1776,New Health Options Act of 2025,Bill,[],119,United States,lower,HR,Bill,NaN,NaN
1,ocd-bill/757c0231-d2d1-408a-a06a-202e3d0abef1,S 693,DHS Better Ballistic Body Armor Act,Bill,[],119,United States,upper,S,Bill,NaN,NaN
2,ocd-bill/be153c8b-cb6d-4bc2-a4e8-b4f2474a309a,S 690,Overdose RADAR Act,Bill,[],119,United States,upper,S,Bill,NaN,NaN
3,ocd-bill/aae01c67-1682-43b0-b4d5-86862e3dc2d7,HR 693,Restoring Confidence in the World Anti-Doping ...,Bill,[],119,United States,lower,HR,Bill,NaN,NaN
4,ocd-bill/c3944c75-caca-4e65-ae2d-910575f0ba68,HR 1806,Triple-Negative Breast Cancer Research and Edu...,Bill,[],119,United States,lower,HR,Bill,NaN,NaN


In [ ]:
df['bill_id'].isna().sum()

np.int64(6598)

In [66]:
df[df['bill_id'].notna()]

,id,identifier,title,classification,subject,session_identifier,jurisdiction,organization_classification,prefix_category,measure_type,bill_id,classification_clean
1618,ocd-bill/f973997d-0aa2-444f-b472-d43b5fff4704,S 146,TAKE IT DOWN Act,Bill,[],119,United States,upper,S,Bill,ocd-bill/f973997d-0aa2-444f-b472-d43b5fff4704,[became-law]
2597,ocd-bill/9a609605-341c-4ebe-a9be-b49ada1f0844,HJRES 25,Providing for congressional disapproval under ...,Resolution,[],119,United States,lower,HJRES,Joint Resolution,ocd-bill/9a609605-341c-4ebe-a9be-b49ada1f0844,[became-law]
3144,ocd-bill/30325e59-e9c1-4faa-951c-d56b5a89eece,HJRES 20,Providing for congressional disapproval under ...,Resolution,[],119,United States,lower,HJRES,Joint Resolution,ocd-bill/30325e59-e9c1-4faa-951c-d56b5a89eece,[became-law]
3429,ocd-bill/64132b9b-fdb2-4e7d-b1f5-41cd1d38cdc0,S 5,Laken Riley Act,Bill,[],119,United States,upper,S,Bill,ocd-bill/64132b9b-fdb2-4e7d-b1f5-41cd1d38cdc0,[became-law]
4174,ocd-bill/f56c4d76-0185-4493-932a-5894e6dbddcb,HJRES 24,Providing for congressional disapproval under ...,Resolution,[],119,United States,lower,HJRES,Joint Resolution,ocd-bill/f56c4d76-0185-4493-932a-5894e6dbddcb,[became-law]
4533,ocd-bill/47448845-eb89-4fb5-aea9-a2f70a1fd079,HJRES 35,Providing for congressional disapproval under ...,Resolution,[],119,United States,lower,HJRES,Joint Resolution,ocd-bill/47448845-eb89-4fb5-aea9-a2f70a1fd079,[became-law]
4634,ocd-bill/d14e801e-90b4-4459-96ba-abcc08cd8b54,SJRES 18,A joint resolution disapproving the rule submi...,Resolution,[],119,United States,upper,SJRES,Joint Resolution,ocd-bill/d14e801e-90b4-4459-96ba-abcc08cd8b54,[became-law]
4638,ocd-bill/e8e8cdca-83bf-434d-9a3b-3df5b7291da3,SJRES 28,A joint resolution disapproving the rule submi...,Resolution,[],119,United States,upper,SJRES,Joint Resolution,ocd-bill/e8e8cdca-83bf-434d-9a3b-3df5b7291da3,[became-law]
4739,ocd-bill/e36cb349-1dc3-4a9a-a4f8-e7c89561299b,SJRES 11,A joint resolution providing for congressional...,Resolution,[],119,United States,upper,SJRES,Joint Resolution,ocd-bill/e36cb349-1dc3-4a9a-a4f8-e7c89561299b,[became-law]
5082,ocd-bill/0fd0f647-5ef1-4488-8e72-62eac9e1debf,HJRES 61,Providing for congressional disapproval under ...,Resolution,[],119,United States,lower,HJRES,Joint Resolution,ocd-bill/0fd0f647-5ef1-4488-8e72-62eac9e1debf,[became-law]


In [ ]:
# Generate timestamped output filename
from datetime import datetime
import os

date_str = datetime.now().strftime("%Y-%m-%d")
filename = f"US_119_bills_processed_{date_str}.csv"

# Define absolute output path
output_dir = "/content/drive/Shareddrives/Data Archive/Bridge_Grades/Output_Files/Temps_Files_For_Preprocessing/"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, filename)

# Save cleaned file
df.to_csv(output_path, index=False)

print("Dataset saved to:", output_path)
print("\nMeasure type distribution:")
print(df["measure_type"].value_counts(dropna=False))
print("\nClassification distribution:")
print(df["classification"].value_counts(dropna=False))

Dataset saved to: /content/drive/Shareddrives/Data Archive/Bridge_Grades/Output_Files/Temps_Files_For_Preprocessing/US_119_bills_processed_2025-11-17.csv

Measure type distribution:
measure_type
Bill                     5670
Simple Resolution         741
Joint Resolution          154
Concurrent Resolution      47
Name: count, dtype: int64

Classification distribution:
classification
Bill          5670
Resolution     942
Name: count, dtype: int64
